In [3]:
# Cell 1: Setup
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, FloatType, DoubleType, ArrayType, StringType
import re
import pyarabic.araby as araby

spark = SparkSession.builder \
    .appName("Arabic-AI-Detection-Features") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f" Spark {spark.version}")

✓ Spark 3.5.8


In [4]:
# Cell 2: Load the processed Parquet 
HDFS_PROCESSED = "hdfs://localhost:9000/user/hadoop/arabic-ai-detection/processed.parquet"
df = spark.read.parquet(HDFS_PROCESSED)


print(f"Loaded: {df.count()} rows")
df.printSchema()
df.show(3, truncate=60, vertical=True)

Loaded: 41940 rows
root
 |-- text: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- generation_method: string (nullable = true)
 |-- text_clean: string (nullable = true)
 |-- char_count: integer (nullable = true)
 |-- word_count: integer (nullable = true)

-RECORD 0-------------------------------------------------------------------------
 text              | صور نظام التعليم عند المرأة الأندلسية تستند إلى دراسة دقي... 
 label             | 1                                                            
 source            | openai                                                       
 generation_method | by_polishing                                                 
 text_clean        | صور نظام التعليم عند المراة الاندلسية تستند الي دراسة دقي... 
 char_count        | 960                                                          
 word_count        | 147                                                          
-RECORD 1-------------

In [5]:
# Cell 3: Feature 1 — Number of diacritics



TASHKEEL_CHARS = set("\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652\u0653\u0654\u0655\u0670")

def count_diacritics(text):
    if text is None:
        return 0
    return sum(1 for ch in text if ch in TASHKEEL_CHARS)

diacritics_udf = F.udf(count_diacritics, IntegerType())
df = df.withColumn("f_diacritics", diacritics_udf(F.col("text")))

print("Sample diacritic counts:")
df.select("source", "f_diacritics").groupBy("source").agg(
    F.avg("f_diacritics").alias("avg"),
    F.min("f_diacritics").alias("min"),
    F.max("f_diacritics").alias("max")
).orderBy("source").show()

Sample diacritic counts:


[Stage 5:============================================>              (3 + 1) / 4]

+------+------------------+---+---+
|source|               avg|min|max|
+------+------------------+---+---+
| allam| 1.757868383404864|  0| 51|
| human|4.2194802098235575|  0|365|
|  jais|1.2632331902718168|  0| 88|
| llama|3.9172627563185505|  0|275|
|openai| 4.292799237005245|  0| 58|
+------+------------------+---+---+



In [6]:
# Cell 4: Feature 2 — Number of colons
def count_colons(text):
    if text is None:
        return 0
    return text.count(":") + text.count("：")  

colons_udf = F.udf(count_colons, IntegerType())
df = df.withColumn("f_colons", colons_udf(F.col("text")))

print("Colon counts per source:")
df.groupBy("source").agg(F.avg("f_colons").alias("avg_colons")).orderBy("source").show()

Colon counts per source:


[Stage 8:>                                                          (0 + 4) / 4]

+------+-------------------+
|source|         avg_colons|
+------+-------------------+
| allam| 0.3053171196948021|
| human|0.41797806390081066|
|  jais|0.14639961850262279|
| llama|0.09477825464949928|
|openai|0.07641869337148308|
+------+-------------------+



In [7]:
# Cell 5: Feature 3 — Number of Arabic particles
ARABIC_PARTICLES = {
    # حروف الجر
    "في", "من", "إلى", "على", "عن", "حتى", "منذ", "مذ", "خلال", "بين", "أمام", "خلف",
    # حروف العطف
    "و", "ف", "ثم", "أو", "أم", "بل", "لكن", "كما",
    # حروف النصب
    "أن", "لن", "كي", "إذن",
    # حروف الجزم
    "لم", "لما", "لا",
    # حروف الاستفهام
    "هل", "أ",
    # حروف التوكيد
    "إن", "كأن", "ليت", "لعل",
}

def count_particles(text):
    if text is None:
        return 0
    
    text = araby.strip_tashkeel(text)
    tokens = text.split()
    return sum(1 for w in tokens if w in ARABIC_PARTICLES)

particles_udf = F.udf(count_particles, IntegerType())
df = df.withColumn("f_particles", particles_udf(F.col("text")))

print("Particles per source:")
df.groupBy("source").agg(F.avg("f_particles").alias("avg_particles")).orderBy("source").show()

Particles per source:


[Stage 11:=============================>                            (2 + 2) / 4]

+------+------------------+
|source|     avg_particles|
+------+------------------+
| allam|12.866356700047687|
| human| 16.24928469241774|
|  jais|10.696709585121603|
| llama|14.901406771578445|
|openai|17.738197424892704|
+------+------------------+



In [8]:
# Cell 6: Feature 4 — 1st person markers
FIRST_PERSON_PRONOUNS = {"أنا", "انا", "نحن"}

def count_first_person(text):
    if text is None:
        return 0
    text = araby.strip_tashkeel(text) 
    text = re.sub(r"[إأآ]", "ا", text)
    
    count = 0
    for w in text.split():
        # Standalone 1st-person pronouns
        if w in FIRST_PERSON_PRONOUNS:
            count += 1
            continue
        # Present-tense verb prefixes
        if len(w) >= 3 and w[0] == "ا" and not w[1] == "ل":  
            
            pass  
        # 1st-singular past suffix
        if len(w) >= 4 and w.endswith("ت"):
            count += 1
        # 1st-plural past suffix
        elif len(w) >= 4 and w.endswith("نا"):
            count += 1
    return count

first_person_udf = F.udf(count_first_person, IntegerType())
df = df.withColumn("f_first_person", first_person_udf(F.col("text")))

print("First person markers per source:")
df.groupBy("source").agg(F.avg("f_first_person").alias("avg_1st_person")).orderBy("source").show()

First person markers per source:


[Stage 14:===========================================>              (3 + 1) / 4]

+------+------------------+
|source|    avg_1st_person|
+------+------------------+
| allam| 4.971983786361469|
| human| 6.441463996185027|
|  jais|3.8502622794468286|
| llama| 5.186099189318074|
|openai| 8.153910348116357|
+------+------------------+



In [9]:
# Cell 7: Compare stylometric features: human vs AI
print(" Stylometric features — Human vs AI averages\n")
df.groupBy("label").agg(
    F.avg("f_diacritics").alias("avg_diacritics"),
    F.avg("f_colons").alias("avg_colons"),
    F.avg("f_particles").alias("avg_particles"),
    F.avg("f_first_person").alias("avg_1st_person"),
).orderBy("label").show()

print("Label 0 = Human, Label 1 = AI")

📊 Stylometric features — Human vs AI averages



[Stage 17:===========================================>              (3 + 1) / 4]

+-----+------------------+-------------------+-----------------+-----------------+
|label|    avg_diacritics|         avg_colons|    avg_particles|   avg_1st_person|
+-----+------------------+-------------------+-----------------+-----------------+
|    0|4.2194802098235575|0.41797806390081066|16.24928469241774|6.441463996185027|
|    1|2.8077908917501193|0.15572842155460181|14.05066762041011|5.540563900810682|
+-----+------------------+-------------------+-----------------+-----------------+

Label 0 = Human, Label 1 = AI


In [10]:
# Cell 8: Save
HDFS_FEATURES_BASIC = "hdfs://localhost:9000/user/hadoop/arabic-ai-detection/features_basic.parquet"
df.write.mode("overwrite").option("compression", "snappy").parquet(HDFS_FEATURES_BASIC)
print(f" Saved features 1-4 to {HDFS_FEATURES_BASIC}")
print(f"Columns: {df.columns}")

[Stage 20:===========================================>              (3 + 1) / 4]

✓ Saved features 1-4 to hdfs://localhost:9000/user/hadoop/arabic-ai-detection/features_basic.parquet
Columns: ['text', 'label', 'source', 'generation_method', 'text_clean', 'char_count', 'word_count', 'f_diacritics', 'f_colons', 'f_particles', 'f_first_person']


In [11]:
# Cell 9: Train Word2Vec on cleaned corpus
from pyspark.ml.feature import Word2Vec, Tokenizer


tokenizer = Tokenizer(inputCol="text_clean", outputCol="tokens")
df_tok = tokenizer.transform(df.filter(F.col("text_clean") != ""))


print("Training Word2Vec on corpus")
w2v = Word2Vec(
    vectorSize=100,
    minCount=3,           
    inputCol="tokens",
    outputCol="text_vec", 
    seed=42
)
w2v_model = w2v.fit(df_tok)
print("  Word2Vec trained")
print(f"  Vocabulary size: {w2v_model.getVectors().count()}")

Training Word2Vec on corpus (this takes 1-2 min)...


26/05/18 09:47:01 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/18 09:47:01 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
                                                                                

✓ Word2Vec trained
  Vocabulary size: 63631


In [12]:
# Cell 10: Compute pairwise cosine similarity variance per document


import numpy as np
from pyspark.sql.types import FloatType


word_vectors_df = w2v_model.getVectors().collect()
word_vec_dict = {row.word: np.array(row.vector) for row in word_vectors_df}


broadcast_vec = spark.sparkContext.broadcast(word_vec_dict)

def embedding_similarity_variance(tokens):
    
    if tokens is None or len(tokens) < 2:
        return 0.0
    
    vecs = []
    vec_dict = broadcast_vec.value
    for tok in tokens:
        if tok in vec_dict:
            vecs.append(vec_dict[tok])
    
    if len(vecs) < 2:
        return 0.0
    
    vecs = np.array(vecs)
    
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)  # avoid div by zero
    vecs_norm = vecs / norms
    
    
    sim_matrix = vecs_norm @ vecs_norm.T
   
    n = len(vecs)
    upper = sim_matrix[np.triu_indices(n, k=1)]
    
    return float(np.var(upper))

variance_udf = F.udf(embedding_similarity_variance, FloatType())


df_with_var = df_tok.withColumn("f_embed_variance", variance_udf(F.col("tokens")))

print("Embedding similarity variance per source:")
df_with_var.groupBy("source").agg(
    F.avg("f_embed_variance").alias("avg_variance"),
    F.stddev("f_embed_variance").alias("std_variance"),
).orderBy("source").show()

Embedding similarity variance per source:


[Stage 29:===========================================>              (3 + 1) / 4]

+------+--------------------+--------------------+
|source|        avg_variance|        std_variance|
+------+--------------------+--------------------+
| allam| 0.03566506667161235|0.006802631940958723|
| human| 0.03856800326213248|0.010042036564954945|
|  jais| 0.03527401664734427|0.007733859794555907|
| llama|  0.0365391959923439|0.007486763735065231|
|openai|0.030807742346024864| 0.00459102462358151|
+------+--------------------+--------------------+



In [13]:
# Cell 11: Human vs AI on embedding variance
print("  Feature 5: Embedding Similarity Variance (Human vs AI)\n")
df_with_var.groupBy("label").agg(
    F.avg("f_embed_variance").alias("avg_variance"),
    F.min("f_embed_variance").alias("min"),
    F.max("f_embed_variance").alias("max"),
).orderBy("label").show()

print("Label 0 = Human, Label 1 = AI")
print("\nHypothesis: Higher variance suggests more semantically diverse text (often human).")

📊 Feature 5: Embedding Similarity Variance (Human vs AI)



[Stage 32:===========================================>              (3 + 1) / 4]

+-----+-------------------+-----------+----------+
|label|       avg_variance|        min|       max|
+-----+-------------------+-----------+----------+
|    0|0.03856800326213248|0.021653224|0.11871058|
|    1|0.03457150541433134|        0.0|0.18153277|
+-----+-------------------+-----------+----------+

Label 0 = Human, Label 1 = AI

Hypothesis: Higher variance suggests more semantically diverse text (often human).


In [14]:
# Cell 12: Save
HDFS_FEATURES_STYLO = "hdfs://localhost:9000/user/hadoop/arabic-ai-detection/features_stylometric.parquet"

df_with_var.drop("tokens").write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(HDFS_FEATURES_STYLO)

print(f"  All 5 stylometric features saved to {HDFS_FEATURES_STYLO}")

# Verify
df_check = spark.read.parquet(HDFS_FEATURES_STYLO)
print(f"\nColumns: {df_check.columns}")
print(f"Rows: {df_check.count()}")

✓ All 5 stylometric features saved to hdfs://localhost:9000/user/hadoop/arabic-ai-detection/features_stylometric.parquet

Columns: ['text', 'label', 'source', 'generation_method', 'text_clean', 'char_count', 'word_count', 'f_diacritics', 'f_colons', 'f_particles', 'f_first_person', 'f_embed_variance']
Rows: 41940


In [15]:
# Cell 13: Summary table of all stylometric features by class
print("=" * 70)
print(" STYLOMETRIC FEATURES — Final Summary (Human vs AI)")
print("=" * 70)

summary = df_with_var.groupBy("label").agg(
    F.avg("f_diacritics").alias("diacritics"),
    F.avg("f_colons").alias("colons"),
    F.avg("f_particles").alias("particles"),
    F.avg("f_first_person").alias("first_person"),
    F.avg("f_embed_variance").alias("embed_variance"),
).orderBy("label")

summary.show(truncate=False)

# Save 
summary.toPandas().to_csv("../reports/figures/stylometric_features_summary.csv", index=False)
print(" Saved summary to reports/figures/stylometric_features_summary.csv")

📊 STYLOMETRIC FEATURES — Final Summary (Human vs AI)


+-----+------------------+-------------------+-----------------+-----------------+-------------------+
|label|diacritics        |colons             |particles        |first_person     |embed_variance     |
+-----+------------------+-------------------+-----------------+-----------------+-------------------+
|0    |4.2194802098235575|0.41797806390081066|16.24928469241774|6.441463996185027|0.03856800326213248|
|1    |2.8077908917501193|0.15572842155460181|14.05066762041011|5.540563900810682|0.03457150541433134|
+-----+------------------+-------------------+-----------------+-----------------+-------------------+



✓ Saved summary to reports/figures/stylometric_features_summary.csv


In [16]:
# Cell 14: TF-IDF features using Spark MLlib
from pyspark.ml.feature import HashingTF, IDF, Tokenizer, StopWordsRemover
from pyspark.ml import Pipeline


HDFS_FEATURES_STYLO = "hdfs://localhost:9000/user/hadoop/arabic-ai-detection/features_stylometric.parquet"
df_features = spark.read.parquet(HDFS_FEATURES_STYLO)

# Build TF-IDF pipeline:

tokenizer = Tokenizer(inputCol="text_clean", outputCol="words")
hashing_tf = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=10000)
idf = IDF(inputCol="raw_features", outputCol="tfidf_features", minDocFreq=5)

tfidf_pipeline = Pipeline(stages=[tokenizer, hashing_tf, idf])

print("Fitting TF-IDF pipeline ")
tfidf_model = tfidf_pipeline.fit(df_features)
df_tfidf = tfidf_model.transform(df_features)

print(" TF-IDF features computed")
print(f"Feature dimensionality: 10,000 (hashed)")
df_tfidf.select("source", "label", "tfidf_features").show(2, truncate=80)

Fitting TF-IDF pipeline (1-2 min)...


✓ TF-IDF features computed
Feature dimensionality: 10,000 (hashed)
+------+-----+--------------------------------------------------------------------------------+
|source|label|                                                                  tfidf_features|
+------+-----+--------------------------------------------------------------------------------+
| human|    0|(10000,[9,64,218,249,352,446,504,758,779,855,1047,1380,1389,1412,1563,1635,17...|
| human|    0|(10000,[75,93,179,211,431,480,789,805,856,869,1003,1033,1092,1261,1267,1323,1...|
+------+-----+--------------------------------------------------------------------------------+
only showing top 2 rows



In [17]:
# Cell 15: Re-attach Word2Vec document vectors 


df_tok_2 = Tokenizer(inputCol="text_clean", outputCol="tokens_w2v").transform(df_tfidf)



w2v_model_clone = Word2Vec(
    vectorSize=100, minCount=3,
    inputCol="tokens_w2v", outputCol="w2v_features", seed=42
).fit(df_tok_2)
df_w2v = w2v_model_clone.transform(df_tok_2)

print(" Word2Vec document vectors attached")
df_w2v.select("source", "label", "w2v_features").show(2, truncate=80)

✓ Word2Vec document vectors attached
+------+-----+--------------------------------------------------------------------------------+
|source|label|                                                                    w2v_features|
+------+-----+--------------------------------------------------------------------------------+
| human|    0|[0.00968812857145252,0.10378090679105893,0.060435511190313665,-0.018686228078...|
| human|    0|[0.010024504700759579,0.09807347546934654,0.06499627802318696,0.0018953309641...|
+------+-----+--------------------------------------------------------------------------------+
only showing top 2 rows



In [18]:
# Cell 16: Save 
HDFS_FEATURES_ALL = "hdfs://localhost:9000/user/hadoop/arabic-ai-detection/features_all.parquet"


df_final = df_w2v.drop("words", "raw_features", "tokens_w2v")

df_final.write.mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(HDFS_FEATURES_ALL)

print(f" Full feature set saved to {HDFS_FEATURES_ALL}")
print(f"Final columns: {df_final.columns}")

# Save the trained pipeline models locally 
import os
os.makedirs("../models", exist_ok=True)
tfidf_model.write().overwrite().save("file://" + os.path.abspath("../models/tfidf_pipeline"))
w2v_model_clone.write().overwrite().save("file://" + os.path.abspath("../models/word2vec"))
print(" TF-IDF and Word2Vec models saved to models/")

✓ Full feature set saved to hdfs://localhost:9000/user/hadoop/arabic-ai-detection/features_all.parquet
Final columns: ['text', 'label', 'source', 'generation_method', 'text_clean', 'char_count', 'word_count', 'f_diacritics', 'f_colons', 'f_particles', 'f_first_person', 'f_embed_variance', 'tfidf_features', 'w2v_features']


26/05/18 09:56:51 WARN TaskSetManager: Stage 69 contains a task of very large size (7531 KiB). The maximum recommended task size is 1000 KiB.
[Stage 71:>                                                         (0 + 1) / 1]

✓ TF-IDF and Word2Vec models saved to models/
